# vLLM Dual-GPU MCQ Generation Pipeline

**Environment:** Kaggle T4x2 (2x NVIDIA Tesla T4, 16GB each)  
**Models:** Llama-3.2-3B-Instruct + LoRA adapters (QA & Distractor)  
**Architecture:** GPU 0 -> QA Generation | GPU 1 -> Distractor Generation

In [1]:
import sys, subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'vllm', 'transformers', 'accelerate', 'peft', 'bitsandbytes','tqdm'], check=True)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 279.2/279.2 MB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 48.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.7/211.7 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.5/447.5 MB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.0/14.0 MB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.8/178.8 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
grpcio-tools 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 7.35.1 which is incompatible.
ydata-profiling 4.18.4 requires numba<0.63,>=0.60, but you have numba 0.65.0 which is incompatible.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
google-cloud-bigtable 2.36.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<7.0.0,>=3.20.2, but you have protobuf 7.35.1 which is incompatible.
pylibcudf-cu12 26.2.1 requires cuda-python<13.0,>=12.9.2, but you have cuda-python 13.3.1 which is incompatible.
numba-cuda 0.22.2 requires cuda-core<1.0.0,>=0.3.2, but

CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-q', 'vllm', 'transformers', 'accelerate', 'peft', 'bitsandbytes', 'tqdm'], returncode=0)

In [2]:
import os
import json
import csv
import torch
import random
from pathlib import Path
from datetime import datetime
from dataclasses import dataclass
from collections import Counter
from typing import Optional, List, Dict, Tuple
from vllm import LLM, SamplingParams
from vllm.lora.request import LoRARequest

In [3]:
os.environ["VLLM_LOGGING_LEVEL"] = "ERROR"      
os.environ["VLLM_CONFIGURE_LOGGING"] = "0"        
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

In [4]:
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    HF_TOKEN = secrets.get_secret('HF_TOKEN')
    os.environ['HF_TOKEN'] = HF_TOKEN
    print('Secrets loaded from Kaggle Secrets')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN', '')
    if not HF_TOKEN:
        raise ValueError('HF_TOKEN not found')
    print('Secrets loaded from environment')

n_gpus = torch.cuda.device_count()
assert n_gpus >= 2, f'Need 2 GPUs, got {n_gpus}'

Secrets loaded from Kaggle Secrets


In [5]:
@dataclass
class GenerationConfig:
    # Model paths
    base_model: str = 'meta-llama/Llama-3.2-3B-Instruct'
    qa_adapter_path: str = '/kaggle/input/notebooks/baopv051/mcqs-train-qa-model/qa_model_output'
    distractor_adapter_path: str = '/kaggle/input/notebooks/baopv051/mcqs-train-distractor-model/distractor_model_output'
    input_json: str = '/kaggle/input/datasets/baopv051/nmcnpm/learning_contexts.json'

    # Generation control
    questions_per_chunk: int = 1
    max_total_questions: Optional[int] = 100 # None = no limit
    start_chunk_idx: int = 0
    end_chunk_idx: Optional[int] = None
    min_chunk_tokens: int = 100

    # vLLM engine
    tensor_parallel_size: int = 1
    gpu_memory_utilization: float = 0.85
    max_model_len: int = 4096
    dtype: str = 'half'
    enforce_eager: bool = False

    # Generation params
    qa_temperature: float = 0.7
    qa_top_p: float = 0.9
    qa_max_tokens: int = 512
    dist_temperature: float = 0.8
    dist_top_p: float = 0.95
    dist_max_tokens: int = 256

    # Output
    output_dir: str = 'mcq_results'
    save_interval: int = 10
    batch_size: int = 16

    # Prompt templates
    qa_system_prompt: str = (
        'You are an expert educator. Given a passage, generate {n} clear, distinct question-answer pairs answerable from the passage alone.'
        'When the passage contains mathematical formulas, use LaTeX notation ($...$ for inline, $$...$$ for block) in both questions and answers. '
        'Format each as:\nQuestion: <question>\nAnswer: <answer>\n---'
    )
    dist_system_prompt: str = (
        'You are an expert at creating plausible but incorrect answer choices (distractors) for multiple choice questions.'
        'Generate exactly 3 distractors per question. Use LaTeX notation where appropriate. '
        'Format:\nDistractor 1: <text>\nDistractor 2: <text>\nDistractor 3: <text>'
    )

CONFIG = GenerationConfig()
print('Config ready')
print(f'  Input : {CONFIG.input_json}')
print(f'  Output: {CONFIG.output_dir}')
print(f'  Batch : {CONFIG.batch_size} | Q/chunk: {CONFIG.questions_per_chunk}')

Config ready
  Input : /kaggle/input/datasets/baopv051/nmcnpm/learning_contexts.json
  Output: mcq_results
  Batch : 16 | Q/chunk: 1


In [6]:
with open(CONFIG.input_json, 'r', encoding='utf-8') as f:
    raw_items = json.load(f)

print(f'Loaded {len(raw_items)} items from JSON')

chunks = []
skipped = 0
for i, item in enumerate(raw_items):
    if item.get('token_count', 0) < CONFIG.min_chunk_tokens:
        skipped += 1
        continue
    chunks.append({
        'chunk_id'   : f'chunk_{i:05d}',
        'text'       : item['text'],
        'title'      : item.get('title', ''),
        'keywords'   : item.get('keywords', []),
        'token_count': item.get('token_count', 0),
    })

print(f'Chunks after filter (min {CONFIG.min_chunk_tokens} tokens): {len(chunks)} | skipped: {skipped}')

tok_counts  = [c['token_count'] for c in chunks]
latex_items = sum(1 for c in chunks if '$' in c['text'])
print(f'Token count : min={min(tok_counts)}, max={max(tok_counts)}, avg={sum(tok_counts)/len(tok_counts):.0f}')
print(f'Items with LaTeX: {latex_items}/{len(chunks)}')

chunks = chunks[CONFIG.start_chunk_idx : CONFIG.end_chunk_idx]
if CONFIG.max_total_questions:
    max_chunks = (CONFIG.max_total_questions + CONFIG.questions_per_chunk - 1) // CONFIG.questions_per_chunk
    chunks = chunks[:max_chunks]
print(f'Chunks to process: {len(chunks)} | Est. MCQs: {len(chunks) * CONFIG.questions_per_chunk}')

output_dir = Path(CONFIG.output_dir)
output_dir.mkdir(parents=True, exist_ok=True)

Loaded 1650 items from JSON
Chunks after filter (min 100 tokens): 1650 | skipped: 0
Token count : min=100, max=658, avg=281
Items with LaTeX: 129/1650
Chunks to process: 100 | Est. MCQs: 100


In [7]:
print('Initializing vLLM engines...')

# GPU 0: QA engine
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
llm_qa = LLM(
    model=CONFIG.base_model,
    tensor_parallel_size=CONFIG.tensor_parallel_size,
    gpu_memory_utilization=CONFIG.gpu_memory_utilization,
    max_model_len=CONFIG.max_model_len,
    dtype=CONFIG.dtype,
    enable_lora=True,
    max_lora_rank=16,
    max_loras=1,
    enforce_eager=CONFIG.enforce_eager,
)
qa_lora = LoRARequest('qa_adapter', 1, CONFIG.qa_adapter_path)
print('QA engine ready on GPU 0')

# GPU 1: Distractor engine
os.environ['CUDA_VISIBLE_DEVICES'] = '1'
llm_dist = LLM(
    model=CONFIG.base_model,
    tensor_parallel_size=CONFIG.tensor_parallel_size,
    gpu_memory_utilization=CONFIG.gpu_memory_utilization,
    max_model_len=CONFIG.max_model_len,
    dtype=CONFIG.dtype,
    enable_lora=True,
    max_lora_rank=16,
    max_loras=1,
    enforce_eager=CONFIG.enforce_eager,
)
dist_lora = LoRARequest('dist_adapter', 1, CONFIG.distractor_adapter_path)
print('Distractor engine ready on GPU 1')

# Reset 
os.environ['CUDA_VISIBLE_DEVICES'] = '0,1'

Initializing vLLM engines...
INFO 07-01 14:36:59 [api_utils.py:273] non-default args: {'dtype': 'half', 'max_model_len': 4096, 'gpu_memory_utilization': 0.85, 'disable_log_stats': True, 'enable_lora': True, 'model': 'meta-llama/Llama-3.2-3B-Instruct'}


config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

INFO 07-01 14:37:16 [model.py:598] Resolved architecture: LlamaForCausalLM
WARNING 07-01 14:37:16 [model.py:2063] Casting torch.bfloat16 to torch.float16.
INFO 07-01 14:37:16 [model.py:1725] Using max model len 4096
INFO 07-01 14:37:16 [scheduler.py:252] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 07-01 14:37:16 [vllm.py:1006] Asynchronous scheduling is enabled.
INFO 07-01 14:37:16 [kernel.py:276] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

INFO 07-01 14:40:27 [model_runner.py:281] Loading model from scratch...
ERROR 07-01 14:40:28 [fa_utils.py:177] Cannot use FA version 2 is not supported due to FA2 is only supported on devices with compute capability >= 8
INFO 07-01 14:40:28 [cuda.py:480] Using TRITON_ATTN attention backend out of potential backends: ['TRITON_ATTN', 'FLEX_ATTENTION'].
INFO 07-01 14:40:30 [weight_utils.py:849] Filesystem type for checkpoints: OVERLAY. Checkpoint size: 5.98 GiB. Available RAM: 25.88 GiB.
INFO 07-01 14:40:30 [weight_utils.py:872] Auto-prefetch is disabled because the filesystem (OVERLAY) is not a recognized network FS (NFS/Lustre). If you want to force prefetching, start vLLM with --safetensors-load-strategy=prefetch.


Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]


INFO 07-01 14:40:35 [default_loader.py:430] Loading weights took 5.72 seconds
INFO 07-01 14:40:36 [punica_selector.py:20] Using PunicaWrapperGPU.
INFO 07-01 14:40:36 [model_runner.py:302] Model loading took 6.09 GiB and 9.317039 seconds
WARNING 07-01 14:40:36 [topk_topp_sampler.py:62] FlashInfer top-p/top-k sampling unavailable: unsupported compute capability 7.5; falling back. Set VLLM_USE_FLASHINFER_SAMPLER=0 to silence.
INFO 07-01 14:40:42 [backends.py:1089] Using cache directory: /root/.cache/vllm/torch_compile_cache/fc8affc147/rank_0_0/backbone for vLLM's torch.compile
INFO 07-01 14:40:42 [backends.py:1148] Dynamo bytecode transform time: 5.04 s
INFO 07-01 14:40:45 [backends.py:292] Directly load the compiled graph(s) for compile range (1, 8192) from the cache, took 2.279 s
INFO 07-01 14:40:45 [decorators.py:311] Directly load AOT compilation from path /root/.cache/vllm/torch_compile_cache/torch_aot_compile/667644c57c5869554a218ee3135b679c245a74c7d0fbfa346326dcd5fa81d92c/rank_0_0/

Capturing CUDA graphs (FULL): 100%|██████████| 70/70 [00:14<00:00,  4.76it/s]


INFO 07-01 14:42:10 [model_runner.py:726] Graph capturing finished in 69 secs, took 1.07 GiB
INFO 07-01 14:42:26 [jit_monitor.py:60] Kernel JIT monitor activated — Triton JIT compilations during inference will be logged as warnings.
INFO 07-01 14:42:26 [core.py:337] init engine (profile, create kv cache, warmup model) took 109.93 s (compilation: 7.84 s)
INFO 07-01 14:42:26 [kernel.py:276] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])
Distractor engine ready on GPU 1


In [8]:
qa_sampling = SamplingParams(
    temperature=CONFIG.qa_temperature,
    top_p=CONFIG.qa_top_p,
    max_tokens=CONFIG.qa_max_tokens,
    stop=['<|eot_id|>'],
)
dist_sampling = SamplingParams(
    temperature=CONFIG.dist_temperature,
    top_p=CONFIG.dist_top_p,
    max_tokens=CONFIG.dist_max_tokens,
    stop=['<|eot_id|>'],
)

In [9]:
class QAGenerator:

    def __init__(self, llm_engine, lora_request, config: GenerationConfig):
        self.llm = llm_engine
        self.lora = lora_request
        self.config = config

    def _build_prompt(self, context: str, title: str, n: int) -> str:
        sys_p = self.config.qa_system_prompt.format(n=n)
        user_content = f'Title: {title}\nPassage:\n{context}' if title else f'Passage:\n{context}'
        return (
            '<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n'
            f'{sys_p}\n'
            '<|eot_id|><|start_header_id|>user<|end_header_id|>\n\n'
            f'{user_content}\n'
            '<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n'
        )

    def _parse(self, output: str) -> List[Dict]:
        pairs, current_q, current_a = [], None, None
        for line in output.strip().split('\n'):
            line = line.strip()
            if line.lower().startswith('question:'):
                if current_q and current_a:
                    pairs.append({'question': current_q, 'answer': current_a})
                current_q = line.split(':', 1)[1].strip()
                current_a = None
            elif line.lower().startswith('answer:'):
                current_a = line.split(':', 1)[1].strip()
            elif line == '---':
                if current_q and current_a:
                    pairs.append({'question': current_q, 'answer': current_a})
                current_q = current_a = None
        if current_q and current_a:
            pairs.append({'question': current_q, 'answer': current_a})
        return pairs[:self.config.questions_per_chunk]

    def generate(self, chunks: List[Dict]) -> List[List[Dict]]:
        prompts = [
            self._build_prompt(c['text'], c.get('title', ''), self.config.questions_per_chunk)
            for c in chunks
        ]
        outputs = self.llm.generate(prompts, qa_sampling, lora_request=self.lora, use_tqdm=True)
        return [self._parse(o.outputs[0].text) for o in outputs]


class DistractorGenerator:

    def __init__(self, llm_engine, lora_request, config: GenerationConfig):
        self.llm = llm_engine
        self.lora = lora_request
        self.config = config

    def _build_prompt(self, context: str, question: str, answer: str) -> str:
        return (
            '<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n'
            f'{self.config.dist_system_prompt}\n'
            '<|eot_id|><|start_header_id|>user<|end_header_id|>\n\n'
            f'Passage: {context}\nQuestion: {question}\nCorrect Answer: {answer}\n\n'
            '<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n'
        )

    def _parse(self, output: str) -> List[str]:
        dists = []
        for i in range(1, 4):
            prefix = f'distractor {i}:'
            for line in output.strip().split('\n'):
                if line.lower().strip().startswith(prefix):
                    dists.append(line.split(':', 1)[1].strip())
                    break
        while len(dists) < 3:
            dists.append('[N/A]')
        return dists[:3]

    def generate(self, items: List[Dict]) -> List[List[str]]:
        prompts = [self._build_prompt(i['context'], i['question'], i['answer']) for i in items]
        outputs = self.llm.generate(prompts, dist_sampling, lora_request=self.lora)
        return [self._parse(o.outputs[0].text) for o in outputs]


qa_gen = QAGenerator(llm_qa, qa_lora, CONFIG)
dist_gen = DistractorGenerator(llm_dist, dist_lora, CONFIG)

In [10]:
def shuffle_options(correct: str, distractors: List[str], seed_str: str) -> Tuple[Dict, str]:
    random.seed(hash(seed_str) % (2**32))
    opts_list = [correct] + distractors[:3]
    random.shuffle(opts_list)
    opts = dict(zip(['A', 'B', 'C', 'D'], opts_list))
    key  = next(k for k, v in opts.items() if v == correct)
    return opts, key

def load_processed_chunks(jsonl_path: Path) -> set:
    processed = set()
    if jsonl_path.exists():
        with open(jsonl_path, 'r', encoding='utf-8') as f:
            for line in f:
                if line.strip():
                    try:
                        processed.add(json.loads(line)['chunk_id'])
                    except Exception:
                        pass
    return processed

In [11]:
jsonl_path = output_dir / 'mcq_output.jsonl'
csv_path   = output_dir / 'mcq_output.csv'

# Resume support
processed_chunks = load_processed_chunks(jsonl_path)
if processed_chunks:
    print(f'Resuming: {len(processed_chunks)} chunks already done')
    chunks_to_run = [c for c in chunks if c['chunk_id'] not in processed_chunks]
else:
    chunks_to_run = list(chunks)
print(f'Chunks to process: {len(chunks_to_run)}')

csv_file = open(csv_path, 'a', newline='', encoding='utf-8')
fieldnames = [
    'chunk_id', 'title', 'context', 'question', 'correct_answer',
    'distractor_1', 'distractor_2', 'distractor_3',
    'option_A', 'option_B', 'option_C', 'option_D', 'answer_key',
]
csv_writer = csv.DictWriter(csv_file, fieldnames=fieldnames)
if not processed_chunks:
    csv_writer.writeheader()

total_mcqs, failed_chunks = 0, 0
print(f'Starting: {len(chunks_to_run)} chunks, batch={CONFIG.batch_size}')

for batch_idx in range(0, len(chunks_to_run), CONFIG.batch_size):
    batch = chunks_to_run[batch_idx : batch_idx + CONFIG.batch_size]
    if not batch:
        continue
    batch_num     = batch_idx // CONFIG.batch_size + 1
    total_batches = (len(chunks_to_run) + CONFIG.batch_size - 1) // CONFIG.batch_size
    print(f'[{batch_num}/{total_batches}] {len(batch)} chunks')

    # Stage 1: QA generation 
    qa_results = qa_gen.generate(batch)

    # Flatten for distractor stage
    dist_inputs, qa_map = [], []
    for ci, (chunk, qa_list) in enumerate(zip(batch, qa_results)):
        for qa in qa_list:
            dist_inputs.append({'context': chunk['text'], 'question': qa['question'], 'answer': qa['answer']})
            qa_map.append((ci, chunk['chunk_id'], chunk.get('title', ''), chunk['text'], qa))

    if not dist_inputs:
        print(f'  No QA pairs from batch {batch_num}')
        failed_chunks += len(batch)
        continue

    # Stage 2: Distractor generation 
    dist_results = dist_gen.generate(dist_inputs)

    # Stage 3: Save results
    with open(jsonl_path, 'a', encoding='utf-8') as jf:
        for (ci, chunk_id, title, context, qa), distractors in zip(qa_map, dist_results):
            opts, ans_key = shuffle_options(qa['answer'], distractors, chunk_id + qa['question'])
            mcq = {
                'chunk_id'      : chunk_id,
                'title'         : title,
                'context'       : context,
                'question'      : qa['question'],
                'correct_answer': qa['answer'],
                'distractors'   : distractors,
                'options'       : opts,
                'answer_key'    : ans_key,
                'metadata': {
                    'timestamp'         : datetime.now().isoformat(),
                    'chunk_token_count' : batch[ci].get('token_count', 0),
                    'keywords'          : batch[ci].get('keywords', []),
                }
            }
            jf.write(json.dumps(mcq, ensure_ascii=False) + '\n')
            csv_writer.writerow({
                'chunk_id'      : chunk_id,
                'title'         : title,
                'context'       : context[:500],
                'question'      : qa['question'],
                'correct_answer': qa['answer'],
                'distractor_1'  : distractors[0],
                'distractor_2'  : distractors[1],
                'distractor_3'  : distractors[2],
                'option_A'      : opts.get('A', ''),
                'option_B'      : opts.get('B', ''),
                'option_C'      : opts.get('C', ''),
                'option_D'      : opts.get('D', ''),
                'answer_key'    : ans_key,
            })
            total_mcqs += 1

    csv_file.flush()
    if batch_num % CONFIG.save_interval == 0:
        print(f'  Progress: {total_mcqs} MCQs saved')

csv_file.close()
print(f'Done: {total_mcqs} MCQs, {failed_chunks} failed chunks')
print(f'JSONL: {jsonl_path}')
print(f'CSV  : {csv_path}')

Chunks to process: 100
Starting: 100 chunks, batch=16
[1/7] 16 chunks


Rendering prompts:   0%|          | 0/16 [00:00<?, ?it/s]

WARNING 07-01 14:42:27 [input_processor.py:157] vLLM has deprecated support for supporting different tokenizers for different LoRAs. By default, vLLM uses base model's tokenizer. If you are using a LoRA with its own tokenizer, consider specifying `--tokenizer [lora_path]` to use the LoRA tokenizer.


Processed prompts: 100%|██████████| 16/16 [00:15<00:00,  1.00it/s, est. speed input: 414.26 toks/s, output: 30.99 toks/s]


Rendering prompts:   0%|          | 0/15 [00:00<?, ?it/s]

...

Rendering prompts:   0%|          | 0/4 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 4/4 [00:04<00:00,  1.06s/it, est. speed input: 319.70 toks/s, output: 29.82 toks/s]


Rendering prompts:   0%|          | 0/4 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 4/4 [00:07<00:00,  1.93s/it, est. speed input: 183.00 toks/s, output: 21.56 toks/s]

Done: 87 MCQs, 0 failed chunks
JSONL: mcq_results/mcq_output.jsonl
CSV  : mcq_results/mcq_output.csv


In [12]:
all_mcqs = []
with open(jsonl_path, 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            all_mcqs.append(json.loads(line))

print(f'Total MCQs   : {len(all_mcqs)}')
print(f'Unique chunks: {len(set(m["chunk_id"] for m in all_mcqs))}')

json_out = output_dir / 'mcq_output.json'
with open(json_out, 'w', encoding='utf-8') as f:
    json.dump(all_mcqs, f, ensure_ascii=False, indent=2)
print(f'JSON array   : {json_out} ({json_out.stat().st_size / 1024:.0f} KB)')

dist = Counter(m['answer_key'] for m in all_mcqs)
print('Answer dist  :', dict(dist))

latex_q = sum(1 for m in all_mcqs if '$' in m['question'])
latex_a = sum(1 for m in all_mcqs if '$' in m['correct_answer'])
print(f'MCQs with LaTeX in question: {latex_q}/{len(all_mcqs)}')
print(f'MCQs with LaTeX in answer  : {latex_a}/{len(all_mcqs)}')

summary = {
    'total_mcqs'          : len(all_mcqs),
    'unique_chunks'       : len(set(m['chunk_id'] for m in all_mcqs)),
    'answer_distribution' : dict(dist),
    'latex_in_question'   : latex_q,
    'latex_in_answer'     : latex_a,
    'timestamp'           : datetime.now().isoformat(),
}
with open(output_dir / 'summary.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

Total MCQs   : 87
Unique chunks: 87
JSON array   : mcq_results/mcq_output.json (176 KB)
Answer dist  : {'D': 26, 'B': 18, 'A': 23, 'C': 20}
MCQs with LaTeX in question: 2/87
MCQs with LaTeX in answer  : 1/87
